In [16]:
"""
HomeGuard Security System Simulator
Author: Tamara
Description: A smart home monitoring system that processes sensor readings
             and triggers alerts for security, safety, and comfort issues.
"""

import random
from datetime import datetime
import time

# System configuration
HOME_MODES = ["HOME", "AWAY", "SLEEP"]
ALERT_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

# Current system state
current_mode = "AWAY"

def create_sensor(sensor_id, location, sensor_type, threshold=None):
    """Creates a sensor data structure (dictionary)."""
    sensor = {
        "id": sensor_id,
        "location": location,
        "type": sensor_type,
        "threshold": threshold,
        "current_value": None
    }
    return sensor

def create_alert(severity, message, sensor_id, timestamp):
    """Creates an alert data structure (dictionary)."""
    alert = {
        "severity": severity,
        "message": message,
        "sensor_id": sensor_id,
        "timestamp": timestamp
    }
    return alert

In [17]:
def is_abnormal_reading(sensor, reading_value):
    """Checks if a sensor reading is abnormal based on sensor type and thresholds."""
    sensor_type = sensor["type"]
    
    if sensor_type == "temperature":
        if reading_value < 35 or reading_value > 95:
            return True
    elif sensor_type == "motion":
        if reading_value == True:
            return True
    elif sensor_type == "door":
        if reading_value == "OPENED":
            return True
    elif sensor_type == "smoke":
        if reading_value == "DETECTED":
            return True
            
    return False

def process_reading(sensor, reading_value, system_mode):
    """Processes a sensor reading and determines if alerts are needed."""
    alerts = []
    timestamp = datetime.now().strftime("%H:%M:%S")
    
    # 1. Security alerts
    if system_mode == "AWAY":
        if sensor["type"] == "motion" and reading_value == True:
            alerts.append(create_alert("HIGH", f"SECURITY: Motion detected in {sensor['location']} while in AWAY mode!", sensor["id"], timestamp))
        if sensor["type"] == "door" and reading_value == "OPENED":
            alerts.append(create_alert("HIGH", f"SECURITY: {sensor['location']} opened while in AWAY mode!", sensor["id"], timestamp))
                    
    # 2. Safety alerts
    if sensor["type"] == "temperature":
        if reading_value < 35:
            alerts.append(create_alert("CRITICAL", f"SAFETY: Freezing temperature risk in {sensor['location']} ({reading_value}°F)!", sensor["id"], timestamp))
        elif reading_value > 95:
            alerts.append(create_alert("CRITICAL", f"SAFETY: Equipment failure/overheat risk in {sensor['location']} ({reading_value}°F)!", sensor["id"], timestamp))
                    
    if sensor["type"] == "smoke" and reading_value == "DETECTED":
        alerts.append(create_alert("CRITICAL", f"SAFETY: Smoke detected in {sensor['location']}! Fire risk!", sensor["id"], timestamp))
                
    # 3. Comfort notifications
    if system_mode == "HOME" and sensor["type"] == "temperature":
        if reading_value < 65 or reading_value > 75:
            alerts.append(create_alert("LOW", f"COMFORT: Temperature in {sensor['location']} is outside comfort range ({reading_value}°F).", sensor["id"], timestamp))

    return alerts

def generate_reading(sensor_type):
    """Generates a realistic reading for a sensor type."""
    if sensor_type == "temperature":
        return random.randint(32, 85)
    elif sensor_type == "motion":
        return random.choice([True, False, False, False])
    elif sensor_type == "door":
        return random.choice(["CLOSED", "CLOSED", "CLOSED", "OPENED"])
    elif sensor_type == "smoke":
        return random.choice(["CLEAR", "CLEAR", "CLEAR", "DETECTED"])
    return None

def trigger_alert(alert):
    """Displays an alert to the user."""
    severity_symbol = {
        "LOW": "ℹ️",
        "MEDIUM": "⚠️",
        "HIGH": "🚨",
        "CRITICAL": "🔥"
    }
    symbol = severity_symbol.get(alert["severity"], "⚠️")
    print(f"[ALERT!] {symbol} {alert['severity']}: {alert['message']}")

def log_event(message, timestamp=None):
    """Logs an event to the console."""
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")

In [18]:
class Sensor:
    """Represents a sensor in the HomeGuard system using Object-Oriented Programming."""
    def __init__(self, sensor_id, location, sensor_type, threshold=None):
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.threshold = threshold
        self.current_value = None
    
    def read(self):
        self.current_value = generate_reading(self.type)
        return self.current_value
        
    def isAbnormal(self):
        sensor_dict = {
            "id": self.id,
            "location": self.location,
            "type": self.type,
            "threshold": self.threshold
        }
        return is_abnormal_reading(sensor_dict, self.current_value)
    
    def reset(self):
        self.current_value = None
    
    def __str__(self):
        status = "No reading" if self.current_value is None else str(self.current_value)
        return f"{self.id} ({self.location}): {status}"

In [19]:
# --- CHALLENGE 1: Learning Mode ---
class LearningModeTracker:
    """Tracks historical temperature readings to dynamically adjust normal ranges."""
    def __init__(self):
        self.temperature_history = []
        
    def add_reading(self, temp_val):
        self.temperature_history.append(temp_val)
        
    def get_dynamic_threshold(self):
        if not self.temperature_history:
            return (65, 75) # Standaard range
        avg_temp = sum(self.temperature_history) / len(self.temperature_history)
        # Dynamische marge van +/- 10 graden rondom het historisch gemiddelde
        return (max(32, avg_temp - 10), min(100, avg_temp + 10))

In [20]:
# --- CHALLENGE 2: False-Alarm Detection ---
class FalseAlarmFilter:
    """Filters potential false alarms by requiring consecutive abnormal readings."""
    def __init__(self):
        self.warning_counts = {}
        
    def check_with_confirmation(self, sensor_id, is_abnormal):
        """Vereist dat een sensor minimaal 2 keer achter elkaar afwijkt voordat het kritiek wordt."""
        if sensor_id not in self.warning_counts:
            self.warning_counts[sensor_id] = 0
            
        if is_abnormal:
            self.warning_counts[sensor_id] += 1
        else:
            self.warning_counts[sensor_id] = max(0, self.warning_counts[sensor_id] - 1)
            
        # Pas na 2 opeenvolgende meldingen wordt het een bevestigd alarm
        return self.warning_counts[sensor_id] >= 2

In [21]:
# --- CHALLENGE 3: Daily Summary Report ---
def generate_summary_report(all_alerts, sensor_activity_log):
    """Generates a daily summary report of all system activities and alerts."""
    print("\n" + "=" * 50)
    print("📊 HOMEGUARD DAILY SUMMARY REPORT")
    print("=" * 50)
    
    total_alerts = len(all_alerts)
    print(f"Total Alerts Generated: {total_alerts}")
    
    # Tel per severity
    severity_counts = {"LOW": 0, "MEDIUM": 0, "HIGH": 0, "CRITICAL": 0}
    for alert in all_alerts:
        sev = alert["severity"]
        if sev in severity_counts:
            severity_counts[sev] += 1
            
    print("\nBreakdown by Severity:")
    for sev, count in severity_counts.items():
        print(f"  - {sev}: {count}")
        
    # Meest actieve sensor opzoeken
    if sensor_activity_log:
        most_active = max(sensor_activity_log, key=sensor_activity_log.get)
        print(f"\nMost Active Sensor ID: {most_active} ({sensor_activity_log[most_active]} triggers)")
    
    print("\nRecommendations:")
    if severity_counts["CRITICAL"] > 0:
        print("  ⚠️ Critical alerts detected! Check smoke detectors and heating systems immediately.")
    else:
        print("  ✅ System operated within safe parameters overall.")
    print("=" * 50)

In [22]:
# --- CHALLENGE 4: Interconnected Smoke & Multi-Temp Escalation ---
def evaluate_interconnected_alerts(readings_batch, alerts_list):
    """
    Evaluates readings as a batch to find cross-sensor escalations.
    Escalates severity if Smoke is detected AND multiple Temperature sensors are high.
    """
    smoke_detected = False
    high_temp_count = 0
    
    for reading in readings_batch:
        sensor_type = reading.get("type")
        val = reading.get("value")
        
        if sensor_type == "smoke" and val == "DETECTED":
            smoke_detected = True
            
        if sensor_type == "temperature" and isinstance(val, (int, float)):
            if val > 80:
                high_temp_count += 1

    if smoke_detected and high_temp_count >= 2:
        alerts_list.append(create_alert(
            "CRITICAL",
            f"FIRE ESCALATION: Smoke detected alongside {high_temp_count} high temperature sensors! Severe hazard confirmed.",
            "MULTI_SENSOR_FIRE",
            datetime.now().strftime("%H:%M:%S")
        ))
    elif smoke_detected and high_temp_count == 1:
        alerts_list.append(create_alert(
            "HIGH",
            "POTENTIAL FIRE HAZARD: Smoke detected with elevated temperature reading.",
            "MULTI_SENSOR_WARNING",
            datetime.now().strftime("%H:%M:%S")
        ))

    return alerts_list

In [23]:
def run_simulation(duration_minutes=3, system_mode="AWAY"):
    """Runs the HomeGuard security system simulation."""
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")
    
    sensors = [
        Sensor("MOTION_001", "Living Room", "motion"),
        Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
        Sensor("TEMP_002", "Garage", "temperature", threshold=35),  # Added second temp sensor for Challenge 4 escalation
        Sensor("DOOR_001", "Front Door", "door"),
        Sensor("SMOKE_001", "Bedroom", "smoke")
    ]
    
    all_alerts = []
    sensor_activity_log = {}
    
    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")
        
        readings_batch = []
        step_alerts = []
        
        for sensor in sensors:
            reading = sensor.read()
            sensor_activity_log[sensor.id] = sensor_activity_log.get(sensor.id, 0) + 1
            
            if sensor.type == "temperature":
                status = "Normal" if 65 <= reading <= 75 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")
            
            # Collect reading into the batch for Challenge 4
            readings_batch.append({
                "id": sensor.id,
                "type": sensor.type,
                "value": reading,
                "location": sensor.location
            })
            
            sensor_dict = {
                "id": sensor.id,
                "location": sensor.location,
                "type": sensor.type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)
            
            for alert in alerts:
                step_alerts.append(alert)
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")
        
        # Evaluate Challenge 4 cross-sensor batch logic
        step_alerts = evaluate_interconnected_alerts(readings_batch, step_alerts)
        all_alerts.extend(step_alerts)
        
        time.sleep(0.5)
    
    # Generate final report (Challenge 3)
    generate_summary_report(all_alerts, sensor_activity_log)
    
    print("\n" + "=" * 50)
    print("Simulation complete!")
    print("=" * 50)

# Run the simulation
run_simulation(duration_minutes=3, system_mode="AWAY")

=== HomeGuard Security System ===
Mode: AWAY


Time: 11:51:55
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [11:51:55] Sending notification to homeowner...
[READING] Kitchen Temperature: 82°F (Abnormal)
[READING] Garage Temperature: 48°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 11:51:55
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 44°F (Abnormal)
[READING] Garage Temperature: 59°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 11:51:56
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [11:51:56] Sending notification to homeowner...
[READING] Kitchen Temperature: 64°F (Abnormal)
[READING] Garage Temperature: 32°F (Abnormal)
[ALERT!] 🔥 CRITICAL: SAFETY: Freezing temperature risk in Garage (32°F)!
[LOG] [11:51:56] Sending notification to homeowner